In [1]:
import os,sys
import pandas as pd
import math
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
# import plotting packages
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec

# Get the current directory
# os.getcwd()
# Get the parent directory (which is the 'main dir')
#os.path.dirname(current_dir)

code_dir=os.path.dirname(os.path.dirname(os.getcwd()))

# 1. Add the main directory to the import path
sys.path.append(code_dir)

# 2. Change the current working directory to the main directory
os.chdir(code_dir)

# RTM running script

In [3]:
# main_Surrogate_LUT_cases

# Table maker

In [2]:
# linux DeepL server
#!ls /home/dengnan/data/RTM/LUTcases/HG/
!ls /mnt/dengnan/LUTcases/HG/ | wc -l

2400


In [3]:
fdir = "/mnt/dengnan/LUTcases/HG/" 
Fls = os.listdir(fdir)
#targetregex = re.compile(r"Results_case2_COD=(\d+\.?\d*)_Tsurf=300_AOD=0\.0_COD=0\.0_th0=")
Fls = [f for f in Fls if f.startswith('Result')]
#Fls = [f for f in Fls if 'COD=20' in f and 'th0=30' in f]
Fls = np.sort(Fls)
print(len(Fls))
Fls[:5]

2400


array(['Results_case2_RH=0.1_Tsurf=309.2_AOD=0.1243_COD=0.15848931924611134_th0=0.npy',
       'Results_case2_RH=0.1_Tsurf=309.2_AOD=0.1243_COD=0.15848931924611134_th0=15.npy',
       'Results_case2_RH=0.1_Tsurf=309.2_AOD=0.1243_COD=0.15848931924611134_th0=30.npy',
       'Results_case2_RH=0.1_Tsurf=309.2_AOD=0.1243_COD=0.15848931924611134_th0=45.npy',
       'Results_case2_RH=0.1_Tsurf=309.2_AOD=0.1243_COD=0.15848931924611134_th0=60.npy'],
      dtype='<U78')

### each channel srf dw os

In [4]:
from fun_nearealtime_RTM import FY4A_calinu, get_calibration_srf
import numpy as np
import pandas as pd
import os
file_dir = './FY4A_data/'
channels = ['C{:02d}'.format(c) for c in range(1, 6 + 1)]
nu0 = np.arange(2500, 35000, 3)  # Wavenumber grid
nu_channels = FY4A_calinu(nu0, channels, "./FY4A_data/", dnu=3)
# df = pd.DataFrame(columns=channels)

data = np.genfromtxt('data/profiles/ASTMG173.csv', delimiter=',', skip_header=2,  # in wavenumber basis
                    names=['wavelength', 'extraterrestrial', '37tilt', 'direct_circum'])
ref_lam = data['wavelength']  # nm avoid hearder 1
ref_E = data['extraterrestrial']
ref_E_nu = -ref_E * ref_lam ** 2 / 1e7  # W/[m2*nm-1] tp W/[m2*cm-1]

F_dw_os_srf_channel = []
for channel in channels:
    # load calibration data : Spectral Response Func
    srf, nu_channel = get_calibration_srf(channel, file_dir)
    F_dw_os_channel = -np.interp(-nu_channel, -1e7 / ref_lam, ref_E_nu)  # W/[m2*cm-1] to W/cm-1
    F_dw_os_SRF = np.multiply(F_dw_os_channel, srf)
    F_dw_os_srf_channel.append(np.trapz(F_dw_os_SRF, nu_channel))
    # nu_idx = np.nonzero(np.isin(nu_channels, nu_channel))[0]  # fixed 1 April.
    
    # # correct uw
    # uw_cor = np.multiply(uw[nu_idx], srf)
    # uw_channel = np.trapz(uw_cor,nu_channel)
    # df.loc[0, channel] = uw_channel
F_dw_os_srf_channel

[100.56360014402173,
 293.8703639771758,
 146.06104052297425,
 12.06884597258561,
 13.936208329862962,
 18.20438461023419]

In [4]:
np.save('./data/computed/F_dw_os_srf_channel.npy', F_dw_os_srf_channel)

### def


In [5]:
def fy4a_calibration_uw(uw):
    from fun_nearealtime_RTM import FY4A_calinu
    import numpy as np
    import pandas as pd
    import os
    file_dir='./FY4A_data/'
    channels = ['C{:02d}'.format(c) for c in range(1, 6 + 1)]
    nu0 = np.arange(2500, 35000, 3)  # Wavenumber grid
    nu_channels = FY4A_calinu(nu0, channels, "./FY4A_data/", dnu=3)
    df = pd.DataFrame(columns=channels)
    F_dw_os_srf_channel = [100.56360014402173,293.8703639771758,146.06104052297425,
                           12.06884597258561,13.936208329862962,18.20438461023419]

    for i, channel in enumerate(channels):
        srf, nu_channel = get_calibration_srf(channel, file_dir)
        nu_idx = np.nonzero(np.isin(nu0, nu_channel))[0]  # fixed 1 April.
        
        # correct uw
        uw_cor = np.multiply(uw[nu_idx], srf)
        uw_channel = np.trapz(uw_cor,nu_channel)
        # normalize uw_channel
        df.loc[0, channel] = uw_channel #/ F_dw_os_srf_channel[i]
    return df

In [7]:
import numpy as np
import pandas as pd
import re
from tqdm import tqdm

# Step 1: Prepare your columns and lists
channels = ['C{:02d}'.format(c) for c in range(1, 7)]
Tsurf, RH, COD, th0 = [], [], [], [] # AOD,[]

# We'll build a list of dicts, each one a row
data_rows = []

nu0 = np.arange(2500, 35000, 3)

for fl in tqdm(Fls, desc="Processing Radiative Transfer Files"):
    # Load data
    results = dict(np.load(fdir+fl, allow_pickle=True)[0].items())
    #try:
    meta = {}    
    match = re.search(r'Tsurf=([\d.]+)', fl)
    if match: meta['Ta'] = float(match.group(1))

    match = re.search(r'RH=([\d.]+)', fl)
    if match: meta['rh'] = float(match.group(1))

    match = re.search(r'_COD=([\d.]+)', fl)
    if match: meta['COD'] = float(match.group(1))

    match = re.search(r'_th0=([\d.]+)', fl)
    if match: meta['th0'] = float(match.group(1))

    Fdw = results.get('F_dw')  # shape: (len(nu0), )
    meta['dsw'] = np.trapz(Fdw,nu0)
    DNI = results.get('F_dni')
    DHI = results.get('F_dhi')
    meta['dni'] = np.trapz(DNI, nu0)
    meta['dhi'] = np.trapz(DHI, nu0)

    uw = results.get('F_uw')
    df_uw_6channel = fy4a_calibration_uw(uw)
    # Save calibrated 6-channel values into meta
    for ch in df_uw_6channel.columns:
        meta[ch] = df_uw_6channel[ch].values[0]
    
    data_rows.append(meta)
# Step 2: Convert to pandas DataFrame
df = pd.DataFrame(data_rows)
print("\nProcessing Complete. DataFrame created.")

Processing Radiative Transfer Files: 100%|██████████| 2400/2400 [1:32:15<00:00,  2.31s/it]  


Processing Complete. DataFrame created.


### Save the DataFrame to a CSV file

In [8]:
df.describe()

,Ta,rh,COD,th0,dsw,dni,dhi,C01,C02,C03,C04,C05,C06
count,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000
mean,316.700000,0.550000,2.243847,30.000000,633.047400,232.699156,400.348243,16.552927,40.150695,29.473099,0.005722,3.362142,2.141874
std,5.591335,0.287288,2.968751,21.217624,222.926770,225.407513,178.562520,7.348396,18.690331,8.775981,0.002651,0.824615,0.665641
min,309.200000,0.100000,0.000000,0.000000,164.724243,0.086988,120.649432,9.171263,17.456200,13.699736,0.003040,1.723831,0.554784
25%,312.950000,0.300000,0.228014,15.000000,434.949327,17.003034,255.258226,11.459850,28.325240,24.073308,0.004197,2.807117,1.651247
50%,316.700000,0.550000,0.815479,30.000000,649.279494,162.804460,364.057276,13.134712,32.421508,28.727918,0.004811,3.420296,2.132491
75%,320.450000,0.800000,2.879183,45.000000,838.329140,409.937264,534.931378,19.086150,45.868636,32.578790,0.006003,3.692307,2.564000
max,324.200000,1.000000,10.000000,60.000000,1010.268148,813.279520,845.762908,39.772363,98.948483,59.674583,0.023790,5.629756,4.185173


In [9]:
df['dni'] = df['dni']/np.cos(np.deg2rad(df['th0']))

In [10]:
df.to_hdf('FY4A_tool/GPR/LUT/SWRTM_case2_54layers_dnu=3_AOD=0.1243_new2400.h5', key='data', mode='w')

In [4]:
df

,Ta,rh,COD,th0,dsw,dni,dhi,C01,C02,C03,C04,C05,C06
0,233.0,0.1,30.0,0.0,327.933687,0.028924,327.904763,58.183314,158.472082,90.245922,3.872331,6.559637,4.763943
1,233.0,0.1,30.0,15.0,309.455681,3.444812,306.128248,56.308787,153.424154,87.651477,3.832070,6.382096,4.703718
2,233.0,0.1,30.0,30.0,257.599315,5.170807,253.121265,51.468226,139.885102,79.845201,3.525628,5.914509,4.473023
3,233.0,0.1,30.0,45.0,183.263017,4.713664,179.929953,43.270488,115.611040,66.533803,3.043345,5.129482,4.030478
4,233.0,0.1,30.0,60.0,103.656356,2.723771,102.294470,31.317342,81.195640,47.595451,2.270559,3.802505,3.235276
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,268.0,1.0,50.0,0.0,202.764462,0.017493,202.746969,67.633313,181.151520,99.512585,0.087856,6.806766,4.720372
1196,268.0,1.0,50.0,15.0,190.696226,2.052772,188.713400,65.616191,175.173605,96.378853,0.089437,6.644018,4.659857
1197,268.0,1.0,50.0,30.0,159.692281,3.162316,156.953635,59.172565,157.777798,86.926022,0.068931,6.145266,4.443862
1198,268.0,1.0,50.0,45.0,113.690129,2.888308,111.647786,48.974354,128.514220,71.644276,0.056184,5.252056,3.994750


In [13]:
import pandas as pd

# The path to your HDF5 file
file= './FY4A_tool/GPR/LUT/SWRTM_case2_54layers_dnu=3_AOD=0.1243.h5'
df_later = pd.read_hdf(file, key='data')

df

,Ta,rh,COD,th0,dsw,dni,dhi,C01,C02,C03,C04,C05,C06
0,273.0,0.1,0.00,0.0,1079.581777,870.848725,208.733052,8.290284,11.319307,8.632896,0.178257,2.835572,0.638391
1,273.0,0.1,0.00,15.0,1035.624272,863.908460,201.152779,8.342095,11.032558,8.333674,0.165743,2.737765,0.621202
2,273.0,0.1,0.00,30.0,907.562794,827.352670,191.054364,8.010665,10.317580,7.467859,0.140395,2.444487,0.553355
3,273.0,0.1,0.00,45.0,705.056595,753.502786,172.249665,7.554477,9.033406,6.135941,0.111286,1.984709,0.448529
4,273.0,0.1,0.00,60.0,446.853783,610.215019,141.746274,6.843166,7.305283,4.411551,0.065961,1.418724,0.319273
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11395,268.0,1.0,6.31,0.0,778.016673,3.221501,774.795172,22.289869,35.119724,13.271900,0.011567,3.971688,0.901956
11396,268.0,1.0,6.31,15.0,734.922052,16.303393,719.174183,22.211013,35.005497,13.076584,0.014428,3.964810,0.910114
11397,268.0,1.0,6.31,30.0,612.647241,16.219850,598.600439,21.670979,34.129519,12.441839,0.012264,3.748553,0.852294
11398,268.0,1.0,6.31,45.0,436.380152,12.694398,427.403857,20.247160,31.362134,11.220990,0.010134,3.341426,0.779177


In [14]:
df_combined = pd.concat([df, df_later], ignore_index=True)
    
print("\n✅ DataFrames combined successfully.")
print(f"Original df rows: {len(df)}")
print(f"df7200 rows: {len(df)}")
print(f"Combined df_combined rows: {len(df_combined)}")
print(f"Combined DataFrame shape: {df_combined.shape}")



✅ DataFrames combined successfully.
Original df rows: 11400
df7200 rows: 11400
Combined df_combined rows: 15092
Combined DataFrame shape: (15092, 13)


In [15]:
df_combined.to_hdf("./FY4A_tool/GPR/LUT/SWRTM_case2_54layers_dnu=3_AOD=0.1243_new.h5", key='data', mode='w')

In [16]:
df_combined.describe()

,Ta,rh,COD,th0,dsw,dni,dhi,C01,C02,C03,C04,C05,C06
count,15092.000000,15092.000000,15092.000000,15092.000000,15092.000000,15092.000000,15092.000000,15092.000000,15092.000000,15092.000000,15092.000000,15092.000000,15092.000000
mean,260.488073,0.551809,9.332030,29.996024,615.768641,262.755154,394.320766,20.896996,44.182596,23.950202,0.377223,3.401227,1.472427
std,26.478321,0.286319,14.207615,21.215311,313.814456,290.363948,206.046951,16.810890,49.752469,28.175795,0.544493,1.417350,1.534695
min,213.000000,0.100000,0.000000,0.000000,57.174321,0.013968,56.427641,6.832717,7.267078,4.345912,0.000995,1.401151,0.288493
25%,238.000000,0.300000,0.250000,15.000000,345.990227,6.097387,235.418107,8.769001,12.028127,7.775248,0.008820,2.502670,0.564876
50%,263.000000,0.600000,1.580000,30.000000,634.064591,113.987530,335.857985,11.954227,17.259279,9.150407,0.120634,2.910149,0.653605
75%,283.000000,0.800000,10.000000,45.000000,911.797465,525.758201,525.767534,27.819241,44.893834,15.260640,0.592602,3.871361,0.962504
max,303.000000,1.000000,50.000000,60.000000,1138.591306,914.537632,934.209911,69.946653,181.152150,100.981809,3.928597,6.855320,4.793795
